# Telco Customer Churn Prediction
## Logistic Regression - Full Data Science Project

**Position Target:** Banking Data Analysis / FinTech Job Application

**Tech Stack:** Python | pandas | scikit-learn | matplotlib | seaborn | Logistic Regression

**Dataset:** Telco Customer Churn (7,043 records, 21 features, 26.5% churn rate)


## 1. Project Background & Business Understanding

### Why Customer Churn Matters

In banking and telecom, **customer churn** is a core KPI:

- **Acquisition cost >> retention cost:** acquiring a new customer costs 5-7x more than keeping an existing one
- **Profit impact:** studies show reducing churn by 5% can boost profits by 25%-95%
- **Prevention > cure:** identifying at-risk customers early is far cheaper than winning them back

### Analysis Goals

1. **Quantify** how each feature impacts churn probability
2. **Profile** high-risk customers for targeted intervention
3. **Propose** actionable retention strategies backed by data

### Why Logistic Regression?

- **High interpretability:** each coefficient directly shows impact direction and magnitude
- **Banking-compliance friendly:** regulators demand explainable model decisions
- **Strong baseline:** serves as reference point before trying more complex models


## 2. Data Loading & Initial Exploration

In [ ]:
# 导入依赖库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 全局配置
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")
RANDOM_STATE = 42

# 银行蓝 + 警示橙配色
BLUE = (0.122, 0.467, 0.706)
ORANGE = (1.000, 0.498, 0.055)
print("所有依赖库加载成功。")


In [ ]:
# 加载数据集
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv", engine="python")
print(f"形状: {df.shape[0]} 行 x {df.shape[1]} 列")
print(f"内存: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
df.head(10)


In [ ]:
# 各列数据类型
print("=" * 60)
print("列数据类型")
print("=" * 60)
print(df.dtypes.to_string())


In [ ]:
# 数值列统计摘要
df.describe()


### Data Dictionary

| Feature | Type | Description | Values |
|---------|------|-------------|--------|
| customerID | ID | Customer identifier | Drop |
| gender | Binary | Gender | Male / Female |
| SeniorCitizen | Binary | Is senior citizen | 0 / 1 |
| Partner | Binary | Has partner | Yes / No |
| Dependents | Binary | Has dependents | Yes / No |
| 	enure | Numeric | Months with company | 0-72 |
| PhoneService | Binary | Has phone service | Yes / No |
| MultipleLines | Multi | Multiple lines | Yes / No / No phone service |
| InternetService | Multi | Internet service type | DSL / Fiber optic / No |
| OnlineSecurity | Multi | Online security add-on | Yes / No / No internet service |
| OnlineBackup | Multi | Online backup add-on | Yes / No / No internet service |
| DeviceProtection | Multi | Device protection add-on | Yes / No / No internet service |
| TechSupport | Multi | Tech support add-on | Yes / No / No internet service |
| StreamingTV | Multi | Streaming TV add-on | Yes / No / No internet service |
| StreamingMovies | Multi | Streaming movies add-on | Yes / No / No internet service |
| Contract | Multi | Contract type | Month-to-month / One year / Two year |
| PaperlessBilling | Binary | Paperless billing | Yes / No |
| PaymentMethod | Multi | Payment method | 4 types |
| MonthlyCharges | Numeric | Monthly charge (USD) | 18.25-118.75 |
| TotalCharges | Needs cleaning | Total charges | object -> float |
| **Churn** | **Target** | **Customer churned** | **Yes / No** |

> **Note:** TotalCharges is object type; new customers (tenure=0) have empty strings that need conversion.


## 3. Data Cleaning

In [ ]:
# 3.1 检查缺失值
print("各列缺失值数量:")
missing = df.isnull().sum()
print(missing[missing > 0] if (missing > 0).any() else "未发现缺失值。")

# TotalCharges 中的空字符串不算 NaN
print(f"\nTotalCharges 空字符串: {(df['TotalCharges'] == ' ').sum()} 行")
print(f"对应 tenure=0 的客户: {(df['tenure'] == 0).sum()} 行")


In [ ]:
# 3.2 删除 ID 列 + 清洗 TotalCharges
df_clean = df.copy()
df_clean.drop('customerID', axis=1, inplace=True)

# TotalCharges: string -> float, 空字符串 -> NaN -> 0
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
nan_count = df_clean['TotalCharges'].isnull().sum()
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0.0)
print(f"TotalCharges 清洗完成: {nan_count} 个空值 (tenure=0) -> 填充为 0")
print(f"清洗后数据集形状: {df_clean.shape}")

# 3.3 检查重复值
dup_count = df_clean.duplicated().sum()
print(f"重复行: {dup_count}" if dup_count > 0 else "未发现重复行。")


In [ ]:
# 3.4 编码目标变量
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})
churn_counts = df_clean['Churn'].value_counts()
print("流失分布:")
print(f"  未流失 (0): {churn_counts[0]:5d}  ({churn_counts[0]/len(df_clean)*100:.2f}%)")
print(f"  已流失 (1): {churn_counts[1]:5d}  ({churn_counts[1]/len(df_clean)*100:.2f}%)")


## 4. Exploratory Data Analysis (EDA)

Each chart includes business interpretation highlighting **which features strongly correlate with churn**.


In [ ]:
# 4.1 流失比例饼图
fig, ax = plt.subplots(figsize=(6, 5))
counts = df_clean['Churn'].value_counts()
ax.pie(counts.values, labels=['未流失', '已流失'], autopct='%1.1f%%',
       colors=[BLUE, ORANGE], startangle=90, explode=(0, 0.05),
       textprops={'fontsize': 13})
ax.set_title('客户流失分布', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("业务解读: 数据集流失率 26.5%，属于中等偏度不平衡数据（约 1:2.8）。")
print("无需 SMOTE 过采样，模型可直接处理。")


In [ ]:
# 4.2 数值特征分布（按流失状态分组）
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(numeric_cols):
    for churn_val, color, label in [(0, BLUE, '未流失'), (1, ORANGE, '已流失')]:
        subset = df_clean[df_clean['Churn'] == churn_val][col].dropna().values
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=label)
    axes[i].set_title(f'{col} 按流失状态', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('数量')
    axes[i].legend()
plt.tight_layout()
plt.show()

print("业务解读:")
print("- tenure: 流失客户集中在 0-10 个月（新客户前几个月是流失高危期）")
print("- MonthlyCharges: 月费 USD 70-100 区间流失率最高")
print("- TotalCharges: 低总消费客户更容易流失（与短 tenure 一致）")


In [ ]:
# 4.3 分类特征流失率对比
cat_features = ['Contract', 'InternetService', 'PaymentMethod',
                'gender', 'SeniorCitizen', 'Partner', 'Dependents',
                'PhoneService', 'PaperlessBilling', 'TechSupport']
n_rows = 5
fig, axes = plt.subplots(n_rows, 2, figsize=(14, 20))
axes = axes.flatten()

for i, feat in enumerate(cat_features):
    churn_rate = df_clean.groupby(feat)['Churn'].mean().sort_values(ascending=False) * 100
    bars = axes[i].bar(range(len(churn_rate)), churn_rate.values,
                       color=[BLUE if v < 30 else ORANGE for v in churn_rate.values])
    axes[i].set_xticks(range(len(churn_rate)))
    axes[i].set_xticklabels(churn_rate.index, rotation=30, ha='right', fontsize=9)
    axes[i].set_title(f'{feat} 流失率', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('流失率 (%)')
    axes[i].set_ylim(0, max(churn_rate.values) * 1.3)
    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

plt.suptitle('各分类特征的流失率对比', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("业务解读:")
print("- Contract: Month-to-month 合同流失率 42.7%，远高于两年合同的 2.8%（10 倍差距！）")
print("- InternetService: Fiber optic 流失率 41.9%，DSL 仅 19.0%")
print("- PaymentMethod: Electronic check 流失率 45.3%，自动扣款最低")
print("- TechSupport: 无技术支持客户流失率 41.6%，有技术支持仅 15.2%")
print("- 有伴侣 (Partner) 和家属 (Dependents) 的客户流失率显著更低")


In [ ]:
# 4.4 相关性热力图
corr_df = df_clean.select_dtypes(include=[np.number])
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            vmin=-1, vmax=1, center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('特征相关性热力图', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("业务解读:")
print("- tenure 与 TotalCharges 高度正相关 (r=0.83)，符合常理")
print("- 特征间未出现极强共线性（|r| > 0.9），无需额外处理")


In [ ]:
# 4.5 在网时长 vs 月费散点图
fig, ax = plt.subplots(figsize=(8, 6))
for churn_val, color, label, marker in [(0, BLUE, '未流失', 'o'), (1, ORANGE, '已流失', 'x')]:
    subset = df_clean[df_clean['Churn'] == churn_val]
    ax.scatter(subset['tenure'], subset['MonthlyCharges'],
               c=color, label=label, marker=marker, alpha=0.4, s=20)
ax.set_xlabel('在网月数', fontsize=12)
ax.set_ylabel('月消费 (USD)', fontsize=12)
ax.set_title('在网时长 vs 月消费 — 按流失状态', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("业务解读:")
print("- 高风险区域：月费 > USD 70 且 tenure < 12 个月")
print("- 长在网 + 高消费客户很少流失（右下角密集区域）")
print("- 建议：重点对高月费新客户进行干预")


## 5. Feature Engineering

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 5.1 二分类特征 Label Encode
# "No internet service" / "No phone service" 等价于 No，统一编码为 0
binary_mappings = {
    'gender': {'Male': 1, 'Female': 0},
    'Partner': {'Yes': 1, 'No': 0},
    'Dependents': {'Yes': 1, 'No': 0},
    'PhoneService': {'Yes': 1, 'No': 0},
    'PaperlessBilling': {'Yes': 1, 'No': 0},
    'OnlineSecurity': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'OnlineBackup': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'DeviceProtection': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'TechSupport': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'StreamingTV': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'StreamingMovies': {'Yes': 1, 'No': 0, 'No internet service': 0},
    'MultipleLines': {'Yes': 1, 'No': 0, 'No phone service': 0},
}

df_fe = df_clean.copy()
for col, mapping in binary_mappings.items():
    df_fe[col] = df_fe[col].map(mapping)

# 5.2 多分类特征 One-Hot Encode
multi_cat_cols = ['InternetService', 'Contract', 'PaymentMethod']
df_fe = pd.get_dummies(df_fe, columns=multi_cat_cols, drop_first=False, dtype=np.float64)

print(f"编码后: {df_fe.shape[1]} 个特征 ({df_fe.shape[1]-1} 个预测变量 + 目标)")

# 5.3 StandardScaler 标准化
feature_cols = [c for c in df_fe.columns if c != 'Churn']
scaler = StandardScaler()
df_fe[feature_cols] = scaler.fit_transform(df_fe[feature_cols])
print(f"StandardScaler 已应用于 {len(feature_cols)} 个特征")

# 5.4 训练/测试集划分（80/20，分层抽样）
X = df_fe.drop('Churn', axis=1)
y = df_fe['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\n训练集: {X_train.shape[0]} 样本，流失率: {y_train.mean()*100:.1f}%")
print(f"测试集: {X_test.shape[0]} 样本，流失率: {y_test.mean()*100:.1f}%")


## 6. Logistic Regression Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# 6.1 GridSearchCV 超参数调优
# 搜索 C（正则化强度倒数）的最佳值
params = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'max_iter': [2000],
    'solver': ['lbfgs'],
}
lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=2000)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(lr, params, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print(f"\n最佳参数:     {grid.best_params_}")
print(f"最佳 CV ROC-AUC: {grid.best_score_:.4f}")
model = grid.best_estimator_


In [ ]:
# 6.2 系数提取与排序
coef_df = pd.DataFrame({
    '特征': X.columns,
    '系数': model.coef_[0],
    '胜率比': np.exp(model.coef_[0]),
    '系数绝对值': np.abs(model.coef_[0]),
}).sort_values('系数绝对值', ascending=False).reset_index(drop=True)

# 展示 Top 15
print(f"{'排名':<5} {'特征':<35} {'系数':>10} {'胜率比':>10} {'方向':>15}")
print("-" * 80)
for i, row in coef_df.head(15).iterrows():
    direction = "↑ 增加流失" if row['系数'] > 0 else "↓ 降低流失"
    print(f"{i+1:<5} {row['特征']:<35} {row['系数']:>10.4f} {row['胜率比']:>10.4f} {direction:>15}")


In [ ]:
# 6.3 系数可视化
top = coef_df.head(15).iloc[::-1]  # 反转顺序，最大值在上
colors = [ORANGE if c > 0 else BLUE for c in top['系数'].values]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(top)), top['系数'].values, color=colors)
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top['特征'].values, fontsize=10)
ax.set_xlabel('系数 (对数胜率)', fontsize=12)
ax.set_title('Logistic Regression - Top 15 特征系数', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)

# 在条形旁标注系数值
for bar, val in zip(bars, top['系数'].values):
    label_pos = bar.get_width() + 0.03 if val >= 0 else bar.get_width() - 0.1
    ax.text(label_pos, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=ORANGE, label='正系数 (增加流失风险)'),
    Patch(facecolor=BLUE, label='负系数 (降低流失风险)'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

print("系数解读:")
print("- 蓝色（负系数）= 保护因素，降低流失风险")
print("- 橙色（正系数）= 危险因素，增加流失概率")
print("- 胜率比表示特征每增加 1 标准差，流失胜率（odds）的倍数变化")


## 7. Model Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# 7.1 预测 & 核心指标
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
auc       = roc_auc_score(y_test, y_prob)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("=" * 55)
print("  模型性能 — Logistic Regression")
print("=" * 55)
print(f"  准确率 (Accuracy):   {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  精确率 (Precision):  {precision:.4f}  ({precision*100:.2f}%)")
print(f"  召回率 (Recall):     {recall:.4f}  ({recall*100:.2f}%)")
print(f"  F1 分数:             {f1:.4f}")
print(f"  ROC-AUC:             {auc:.4f}")
print(f"\n  混淆矩阵:")
print(f"  TP={tp:4d} (真正)  FP={fp:4d} (假正)")
print(f"  FN={fn:4d} (假负)  TN={tn:4d} (真负)")
print(f"\n分类报告:")
print(classification_report(y_test, y_pred, target_names=['未流失 (0)', '已流失 (1)']))


In [ ]:
# 7.2 混淆矩阵热力图
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt='d', cmap='Blues',
            xticklabels=['预测未流失', '预测已流失'],
            yticklabels=['实际未流失', '实际已流失'],
            annot_kws={'fontsize': 14}, cbar=False, linewidths=1)
ax.set_title('Logistic Regression - 混淆矩阵', fontsize=14, fontweight='bold')
ax.set_ylabel('实际', fontsize=12)
ax.set_xlabel('预测', fontsize=12)
plt.tight_layout()
plt.show()

print(f"业务解读:")
print(f"- 模型正确分类了 {(tp+tn)/len(y_test)*100:.1f}% 的测试客户")
print(f"- 召回率 {recall*100:.1f}%: 能找出 {recall*100:.0f}% 的真正流失客户")
print(f"- 精确率 {precision*100:.1f}%: 预测为流失的客户中 {precision*100:.0f}% 确实流失")
print(f"- 假阴性 FN={fn}: 漏判的流失客户（最需关注的业务损失）")


In [ ]:
# 7.3 ROC 曲线
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color=ORANGE, linewidth=2.5, label=f'Logistic Regression (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='随机分类器 (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.1, color=ORANGE)
ax.set_xlabel('假正率 (False Positive Rate)', fontsize=12)
ax.set_ylabel('真正率 (True Positive Rate)', fontsize=12)
ax.set_title('ROC 曲线 — Logistic Regression', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.show()

print(f"AUC = {auc:.4f}")
print("AUC 解读: 0.84 表示模型有良好的区分能力——随机选一对客户（一流失一留存），")
print("模型有 84% 的概率给流失客户更高的流失评分。")


## 8. Business Insights & Recommendations

### 8.1 High-Risk Customer Profile

Based on LR coefficients and EDA, the **high-risk churn customer** looks like:

| Dimension | High-Risk | Low-Risk |
|-----------|-----------|----------|
| **Contract** | Month-to-month | Two year |
| **Internet** | Fiber optic | No internet / DSL |
| **Payment** | Electronic check | Bank transfer / Credit card (auto) |
| **Tech Support** | Not subscribed | Subscribed |
| **Tenure** | < 12 months | > 36 months |
| **Monthly Charge** | USD 70-100 | Both extremes |
| **Family Status** | Single, no dependents | Has partner + dependents |

### 8.2 Coefficient Validation

LR coefficients align perfectly with EDA findings:

- **tenure (-1.30):** Each 1-SD increase in tenure reduces churn log-odds by 1.30 -> **loyal customers stay**
- **Contract_Two year (-0.32):** Two-year contracts significantly reduce churn -> **long-term contracts are effective retention tools**
- **InternetService_Fiber optic (+1.13):** Fiber users have the highest churn risk -> **possible service-price mismatch**
- **TechSupport group:** Customers with tech support consistently show lower churn

### 8.3 Actionable Retention Strategies

1. **Month-to-month customers** -> Offer "Sign 1 year, get 15% off" upgrade incentive
2. **Fiber + high monthly charge** -> Proactive network quality check + free tech support trial
3. **New customers (tenure < 6 months)** -> Dedicated onboarding team, first-week callback
4. **Electronic check payers** -> Promote auto credit card/bank payment, offer USD 5 first-month credit
5. **No add-on service users** -> Free 1-month trial of OnlineSecurity / TechSupport

### 8.4 Limitations & Next Steps

| Limitation | Improvement |
|------------|-------------|
| Linear model misses interactions | Try Random Forest / XGBoost |
| Basic feature engineering | Create interaction features (tenure x MonthlyCharges) |
| Structured data only | Add call center logs, support tickets |
| Threshold not optimized | Tune based on business cost (retention cost vs churn loss) |
| Single snapshot | Add time-series data, survival analysis |

---

> **Project Summary:** This project completes the full pipeline from data cleaning, EDA, and feature engineering to Logistic Regression modeling. The model achieves ROC-AUC of 0.84, clearly reveals key churn drivers through interpretable coefficients, and provides actionable business recommendations. Suitable for banking data analysis / FinTech job applications.
